In [ ]:
#pip install ipykernel notebook

In [ ]:
# here we have some text adn we need to just convert that to speech 
# so no voice cloning 
# need to get audio codes out somehow from some model and then use decoder to generate audio ..
# hidden states from LLM  > codec head model to get one codec out 
# get subsequent out from code predictor 
# 1 audio-frame = 32 integers 
# 

In [ ]:
# final modelling layer 
BASE_PATH = '/Users/mohitdulani/Desktop/personal/audio-models/Qwen3-TTS/testing'
import sys
sys.path.insert(0, BASE_PATH)
sys.path.insert(0, '/Users/mohitdulani/Desktop/personal/audio-models/Qwen3-TTS')

import torch 
import transformers
from model import Qwen3TTSForConditionalGeneration , Qwen3TTSTalkerModel 
from model_configuration import Qwen3TTSConfig

from tokenizer import Qwen3TokenizerModel , Qwen3TTSTokenizerV2PreTrainedModel ,Qwen3SpeechTokenizerEncoder , Qwen3TokenizerDecoder
from tokenizer_configuration import Qwen3TTSTokenizerV1EncoderConfig , Qwen3TTSTokenizerV1DecoderConfig

from transformers import MimiConfig
import librosa


In [ ]:
# required things
llm_model = ... 
talker_decoder_model = ... 
first_codebook_generator = ... 
rest_codebook_generator = ... 
tokenizer_decoder = ... 


In [ ]:
#config
mimi_config = MimiConfig()

#dataset 
audio_file = '/Users/mohitdulani/Desktop/personal/audio-models/Qwen3-TTS/testing/dataset/ref_source.wav' 
audio_array, sr = librosa.load(audio_file, sr=mimi_config.sampling_rate)

print(type(audio_array))
print(len(audio_array))
tensor_array = torch.tensor(audio_array)


LLM model (Qwen3TTSForConditionalGeneration)  
Contains all codebook generator, text encoder, outputs final values  

In [ ]:
# load weights for the model 

# from transformers import 
path = '/Users/mohitdulani/.cache/huggingface/hub/models--Qwen--Qwen3-TTS-12Hz-1.7B-Base'
state_dict = torch.load(path)
print(state_dict)

In [ ]:
from transformers import AutoTokenizer

text = "I am mohit and I am testing to see if the tts works or not"
formatted_text = f"<|im_start|>assistant\n{text}<|im_end|>\n<|im_start|>assistant\n"

# tokenizer = AutoTokenizer.from_pretrained("path/to/qwen3-tts")
# input_ids = [tokenizer(formatted_text, return_tensors="pt")["input_ids"]]

# mock input_ids (vocab_size ~151k, seq_len 20) until weights are loaded
input_ids = [torch.randint(0, 151936, (1, 20))]

llm_model = Qwen3TTSForConditionalGeneration(config=Qwen3TTSConfig())

# generate() returns (talker_codes_list, talker_hidden_states_list)
# talker_codes_list : list of tensors, each (T, 32)  — 32 RVQ codebooks per frame
# talker_hidden_states_list : list of tensors, each (T, hidden_size)
code_list, hidden_states = llm_model.generate(
    input_ids=input_ids,
    languages=["auto"],
)


TTS inference 

In [ ]:
text = "I am mohit and I am testing to see if the tts works or not "
texttokenizer = Qwen3TokenizerModel()
hidden_states = llm_model(text)
hidden_states = talker_decoder_model(hidden_states)
first_book = first_codebook_generator(hidden_states)
rest_books = rest_codebook_generator(hidden_states)

final_codebook = torch.cat((first_book, rest_books), dim=1)
speech_tokens = tokenizer_decoder.decode(final_codebook)


In [ ]:
text = 'I am mohit and I am testing to see if the tts works or not'
instruct = 'Say this in a melodic tone and in a cute voice'
hidden_states = llm_model(text)
hidden_states = talker_decoder_model(hidden_states)
first_book = first_codebook_generator(hidden_states)
rest_books = rest_codebook_generator(hidden_states)

final_codebook = torch.cat((first_book, rest_books), dim=1)
speech_tokens = tokenizer_decoder.decode(final_codebook)
